<a href="https://colab.research.google.com/github/donnzgonn95/probable-Jachol/blob/main/Zaawansowany_System_Analityczny_Trading_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
import datetime

# Install Streamlit if it's not already installed
try:
    import streamlit as st
except ImportError:
    !pip install streamlit -qq
    import streamlit as st

st.set_page_config(page_title="Trading AI Analyzer", layout="wide")

# --- MODUŁ POBIERANIA DANYCH ---
def fetch_data(ticker, interval, period):
    try:
        df = yf.download(ticker, interval=interval, period=period, progress=False)
        if df.empty:
            return None
        # Spłaszczenie multi-indexu jeśli występuje w nowych wersjach yfinance
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.dropna(inplace=True)
        return df
    except Exception as e:
        st.error(f"Błąd pobierania danych: {e}")
        return None

# --- MODUŁ TRENDU I MOMENTUM (PRECYZYJNE OBLICZENIA) ---
def calculate_trend_and_adx(df):
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA_200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # Obliczenia True Range (TR) i Directional Movement (DM) dla ADX
    high_diff = df['High'].diff()
    low_diff = df['Low'].diff()

    df['+DM'] = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0.0)
    df['-DM'] = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0.0)

    tr1 = df['High'] - df['Low']
    tr2 = abs(df['High'] - df['Close'].shift(1))
    tr3 = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    atr14 = df['TR'].ewm(alpha=1/14, adjust=False).mean()
    plus_di = 100 * (df['+DM'].ewm(alpha=1/14, adjust=False).mean() / atr14)
    minus_di = 100 * (df['-DM'].ewm(alpha=1/14, adjust=False).mean() / atr14)

    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    df['ADX'] = dx.ewm(alpha=1/14, adjust=False).mean()

    return df

# --- MODUŁ POZIOMÓW WSPARCIA/OPORU (KLASTERYZACJA DBSCAN) ---
def calculate_levels_dbscan(df, atr_multiplier=1.5):
    # Szukanie lokalnych szczytów i dołków z okna 5 świec
    df['Local_High'] = df['High'][(df['High'].shift(1) < df['High']) & (df['High'].shift(-1) < df['High'])]
    df['Local_Low'] = df['Low'][(df['Low'].shift(1) > df['Low']) & (df['Low'].shift(-1) > df['Low'])]

    levels = pd.concat([df['Local_High'].dropna(), df['Local_Low'].dropna()]).values.reshape(-1, 1)

    if len(levels) < 10:
        return []

    # Szacowanie zmienności do promienia klasteryzacji
    recent_atr = df['TR'].tail(14).mean()
    eps = recent_atr * atr_multiplier

    # Algorytm AI - DBSCAN
    clustering = DBSCAN(eps=eps, min_samples=2).fit(levels)
    labels = clustering.labels_

    support_resistance_zones = []
    for label in set(labels):
        if label != -1: # -1 to szum
            zone_prices = levels[labels == label].flatten()
            support_resistance_zones.append(np.mean(zone_prices))

    return sorted(support_resistance_zones)

# --- MODUŁ WOLUMENU (PROFIL I SKOKI) ---
def analyze_volume(df):
    # Wolumen skokowy (np. 2x wyższy niż średnia z 20 okresów)
    df['Vol_SMA_20'] = df['Volume'].rolling(window=20).mean()
    df['Vol_Spike'] = df['Volume'] > (df['Vol_SMA_20'] * 2)

    # Volume Point of Control (VPOC) - Najprostsza wersja
    prices = df['Close'].values
    volumes = df['Volume'].values
    hist, bins = np.histogram(prices, bins=50, weights=volumes)
    max_vol_index = np.argmax(hist)
    vpoc = (bins[max_vol_index] + bins[max_vol_index + 1]) / 2

    return df, vpoc

# --- MODUŁ SYGNAŁÓW I FILTRACJI (PRICE ACTION) ---
def generate_signals(df):
    signals = []

    for i in range(1, len(df)):
        current = df.iloc[i]
        prev = df.iloc[i-1]

        body = abs(current['Close'] - current['Open'])
        total_range = current['High'] - current['Low']

        if total_range == 0: continue

        # 1. Wykrywanie Pin Bar (Pinezka)
        is_bullish_pin = (current['Close'] > current['Open']) and ((current['Open'] - current['Low']) > (2 * body))
        is_bearish_pin = (current['Open'] > current['Close']) and ((current['High'] - current['Open']) > (2 * body))

        # Filtracja Pin Bara
        if is_bullish_pin and current['Vol_Spike'] and current['Close'] > current['EMA_50']:
             signals.append({'Date': df.index[i], 'Type': 'Long', 'Reason': 'Bullish Pin Bar + Vol Spike + Trend Up', 'Price': current['Close']})

        elif is_bearish_pin and current['Vol_Spike'] and current['Close'] < current['EMA_50']:
             signals.append({'Date': df.index[i], 'Type': 'Short', 'Reason': 'Bearish Pin Bar + Vol Spike + Trend Down', 'Price': current['Close']})

        # 2. Wykrywanie Engulfing (Objęcie)
        prev_body = abs(prev['Close'] - prev['Open'])
        is_bullish_engulfing = (prev['Close'] < prev['Open']) and (current['Close'] > current['Open']) and (current['Close'] > prev['Open']) and (current['Open'] < prev['Close'])

        if is_bullish_engulfing and current['ADX'] > 25 and current['EMA_20'] > current['EMA_50']:
            signals.append({'Date': df.index[i], 'Type': 'Long', 'Reason': 'Bullish Engulfing + ADX>25 (Silny Trend)', 'Price': current['Close']})

    return signals

# --- MODUŁ UCZENIA AI (WYKRYWANIE ANOMALII) ---
def detect_anomalies(df):
    # Tworzymy cechy (features) dla modelu
    features = df[['Close', 'Volume']].pct_change().dropna()
    features['TR_norm'] = df['TR'] / df['Close']
    features.dropna(inplace=True)

    # Las Izolacyjny szuka nietypowych wzorców rynkowych
    model = IsolationForest(contamination=0.03, random_state=42)
    features['Anomaly'] = model.fit_predict(features)

    # Przypisanie wyników do głównego DataFrame (-1 to anomalia)
    df['Anomaly'] = 1
    df.loc[features.index, 'Anomaly'] = features['Anomaly']
    return df

# --- INTERFEJS UŻYTKOWNIKA (RAPORTOWANIE) ---
st.title("🧠 System AI dla Analityka Jasnego")
st.markdown("Wykrywanie trendów, stref popytu/podaży, analiza wolumenu i generowanie sygnałów w oparciu o precyzyjne obliczenia.")

col1, col2, col3 = st.columns(3)
with col1:
    ticker = st.text_input("Symbol (np. BTC-USD, AAPL):", "BTC-USD")
with col2:
    interval = st.selectbox("Interwał:", ["15m", "1h", "1d"], index=1)
with col3:
    period = st.selectbox("Zakres danych:", ["5d", "1mo", "3mo", "1y"], index=1)

if st.button("Uruchom Analizę AI"):
    with st.spinner("Pobieranie i przetwarzanie danych..."):
        df = fetch_data(ticker, interval, period)

        if df is not None:
            # Przetwarzanie przez moduły
            df = calculate_trend_and_adx(df)
            levels = calculate_levels_dbscan(df)
            df, vpoc = analyze_volume(df)
            df = detect_anomalies(df)
            signals = generate_signals(df)

            # --- RAPORTOWANIE ---
            current_price = df['Close'].iloc[-1]
            adx_val = df['ADX'].iloc[-1]
            trend_str = "Wzrostowy 📈" if df['EMA_20'].iloc[-1] > df['EMA_50'].iloc[-1] else "Spadkowy 📉"

            st.subheader(f"📊 Raport Analityczny: {ticker}")
            metrics_col1, metrics_col2, metrics_col3, metrics_col4 = st.columns(4)
            metrics_col1.metric("Obecna Cena", f"{current_price:.2f}")
            metrics_col2.metric("Główny Trend (EMA20/50)", trend_str)
            metrics_col3.metric("Siła Trendu (ADX)", f"{adx_val:.2f}", "Silny" if adx_val > 25 else "Boczny (Chaos)")
            metrics_col4.metric("VPOC (Główny Wolumen)", f"{vpoc:.2f}")

            # --- WIZUALIZACJA (WYKRES) ---
            fig = go.Figure()

            # Cena - Świece
            fig.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='Cena'))

            # EMA
            fig.add_trace(go.Scatter(x=df.index, y=df['EMA_20'], line=dict(color='blue', width=1), name='EMA 20'))
            fig.add_trace(go.Scatter(x=df.index, y=df['EMA_50'], line=dict(color='orange', width=1.5), name='EMA 50'))

            # Poziomy wsparcia/oporu z DBSCAN
            for lvl in levels:
                # Pokazujemy tylko te blisko obecnej ceny (np. w obrębie 10%)
                if abs(lvl - current_price) / current_price < 0.10:
                    fig.add_hline(y=lvl, line_dash="dot", annotation_text="Wsparcie/Opór", line_color="gray", opacity=0.5)

            # Anomalie AI (Oznaczone na wykresie)
            anomalies = df[df['Anomaly'] == -1]
            if not anomalies.empty:
                fig.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Low'] - (anomalies['TR']*0.5), mode='markers',
                                         marker=dict(color='purple', size=8, symbol='star'), name='Anomalia AI'))

            # Sygnały wejścia
            for sig in signals:
                color = 'green' if sig['Type'] == 'Long' else 'red'
                symbol = 'triangle-up' if sig['Type'] == 'Long' else 'triangle-down'
                y_pos = df.loc[sig['Date'], 'Low'] * 0.99 if sig['Type'] == 'Long' else df.loc[sig['Date'], 'High'] * 1.01
                fig.add_trace(go.Scatter(x=[sig['Date']], y=[y_pos], mode='markers',
                                         marker=dict(color=color, size=12, symbol=symbol),
                                         name=f"Sygnał: {sig['Reason']}", hovertemplate=sig['Reason']))

            fig.update_layout(title='Wykres Struktury Rynku z Algorytmami AI', xaxis_rangeslider_visible=False, height=600)
            st.plotly_chart(fig, use_container_width=True)

            # --- LISTA SYGNAŁÓW ---
            st.subheader("🔔 Najnowsze wyfiltrowane sygnały")
            if signals:
                recent_signals = sorted(signals, key=lambda x: x['Date'], reverse=True)[:5]
                for s in recent_signals:
                    st.info(f"**{s['Date'].strftime('%Y-%m-%d %H:%M')}** | Typ: **{s['Type']}** | Cena: {s['Price']:.2f} | Powód: {s['Reason']}")
            else:
                st.write("Brak sygnałów spełniających rygorystyczne kryteria filtrowania w tym okresie.")

            st.markdown("---")
            st.markdown("""
            **Jak interpretować wykres jako nowicjusz?**
            1. **Struktura (Średnie):** Gdy niebieska linia (EMA20) jest nad pomarańczową (EMA50) - szukamy tylko sygnałów wzrostowych (Long).
            2. **Fioletowe gwiazdki (AI):** Oznaczają momenty, gdy rynek zachował się nienaturalnie (ogromny skok wolumenu/ceny w krótkim czasie). To często początek nowego impulsu.
            3. **Przerywane linie:** To strefy wyznaczone matematycznie przez algorytm DBSCAN. Obserwuj reakcję ceny na tych poziomach.
            """)

2026-04-30 20:42:06.833 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.845 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-30 20:42:06.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar